1. Install the Vertex AI SDK: Open a terminal window and enter the command below. You can also [install it in a virtualenv](https://googleapis.dev/python/aiplatform/latest/index.html)

In [ ]:
!pip install --upgrade google-genai

## Mobile Industry Expert

### Create an LLM agent that is grounded on Google search. It is a Mobile Industry Expert that will follow strikt rules defined in the system instructions!

- generate(device_name)

In [ ]:
from google import genai
from google.genai import types
import base64
import os
import google.auth

def generate(device_name):
  # Correctly get the default project ID from the environment.
  # google.auth.default() returns (credentials, project_id)

  project_id = "tnn-pnx-consumer-common-ai"

  # Specify the location explicitly. 'us-central1' is a common region for Vertex AI.
  # You might need to change this to your specific Vertex AI region (e.g., 'europe-west1').
  location = "europe-west1"

  client = genai.Client(
      vertexai=True,
      project=project_id,
      location=location,
  )

  si_text1 = """You are an expert in the mobile phone industry, specializing in historical and current device analysis. Your primary task is to categorize and extract relevant features for specific mobile phone models based on their release year and market context. Your output must be optimized for direct ingestion into a BigQuery table, meaning all extracted features should be top-level keys in the JSON object.

Follow these rules strictly:
1.  **Input Parsing:** The user will provide the 'device_name' and 'manufacturer'. You must independently determine the 'release_year' for the specified device name and manufacturer. This 'release_year' is critical for all subsequent analysis.
2.  **Contextual Awareness:** When evaluating a phone, always consider its determined 'release_year'. A 'premium' phone from 2013 must be evaluated against other phones from 2013, not current standards.
3.  **Categorization:** Provide a 'device_series_annual' category for the phone, choosing from: 'Ultra-Premium Flagship', 'Premium Flagship', 'High-End', 'Mid-Range', 'Budget', 'Entry-Level'. If uncertain, state 'Uncategorized'.
4.  **Feature Extraction - Explicit Keys:** Extract the following key technical specifications. Each must be its own top-level key in the JSON output, with a precise value:
    *   **`ram_gb`**: RAM capacity in GB (e.g., 2, 4, 6, 8, 12, 16). If multiple, use the most common/base model.
    *   **`storage_base_gb`**: Base internal storage capacity in GB (e.g., 16, 32, 64, 128, 256, 512, 1024).
    *   **`display_type`**: Primary display technology (e.g., 'Super AMOLED', 'OLED', 'LCD', 'IPS LCD').
    *   **`display_size_inches`**: Display size in inches (e.g., 5.0, 6.1, 6.7).
    *   **`camera_rear_mp`**: Megapixels of the primary rear camera (e.g., 12, 48, 108, 200).
    *   **`camera_front_mp`**: Megapixels of the primary front camera (e.g., 5, 10, 12).
5.  **Price Conversion:** For the 'launch_price', always provide it in NOK (Norwegian Kroner) as the key `launch_price_nok`. If the original launch price is found in another currency, convert it to NOK based on the approximate exchange rate at the time of the phone's release year. State the converted value as a numerical string (e.g., "6000", "12500").
6.  **Context Summary:** Provide a 'context_summary' that briefly explains the phone's market position, key selling points, and relevance in its launch year.
7.  **Conciseness:** Provide direct answers and avoid conversational filler unless explicitly asked.
8.  **Output Format:** Always provide your response in a structured JSON format. Ensure the JSON includes the following top-level keys in this exact order: `device_name`, `manufacturer`, `release_year`, `device_series_annual`, `ram_gb`, `storage_base_gb`, `display_type`, `display_size_inches`, `camera_rear_mp`, `camera_front_mp`, `launch_price_nok`, `context_summary`. All values should be precise.
9.  **Language:** Respond in Norwegian if the user's prompt is in Norwegian, otherwise use English.
10. **User input:** Input Parsing string shall be included as a top-level key in the JSON output.
11. **device_name** Manufacturer should not be included in the device_name.
12. **device_series_annual** Should be one of the following: 'Ultra-Premium Flagship', 'Premium Flagship', 'Flagship' 'High-End', 'Mid-Range', 'Budget', 'Entry-Level'. Example iphone 16 pro max should be categorized as Ultra-Premium Flagship, iphone 16 pro as Premium Flagship, iphone 16 as flagship, iphone se as Mid-Range.
13. **Generation** Should be a number that is defined by the number in its device_series_annual.
14. **device_series_annual_no** Should be a number coresponding to the device_series_annual by these rules excactly: when device_series_annual = 'Uncategorized' then 99
when device_series_annual = 'Entry-Level' then 1
when device_series_annual = 'Budget' then 2
when device_series_annual = 'Mid-Range' then 3
when device_series_annual = 'High-End' then 4
when device_series_annual = 'Flagship' then 5
when device_series_annual = 'Premium Flagship' then 6
when device_series_annual = 'Ultra-Premium Flagship' then 7
{
"""

  model = "gemini-2.5-pro"
  contents = [
    types.Content(
      role="user",
      parts=[
          types.Part(text=device_name)
      ]
    )
  ]
  tools = [
    types.Tool(google_search=types.GoogleSearch()),
  ]

  generate_content_config = types.GenerateContentConfig(
    temperature = 0.5,
    top_p = 0.95,
    seed = 0,
    max_output_tokens = 5000,
    safety_settings = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="OFF"
    )],
    tools = tools,
    system_instruction=[types.Part(text=si_text1)],
    thinking_config=types.ThinkingConfig(
      thinking_budget=-1,
    ),
  )

  response = client.models.generate_content(
    model = model,
    contents = contents,
    config = generate_content_config,
  )
  return response.text

print("The 'generate' function has been updated to accept a device_name and return the result.")

The 'generate' function has been updated to accept a device_name and return the result.


## Create Pandas dataframe

### Call LLM, convert json result and store in dataframe
- get_and_append_phone_data(df, device_name)

In [ ]:
import pandas as pd
import json

def get_and_append_phone_data(df, device_name):
    """Calls the generate function, parses the JSON, normalizes features, and appends it to a DataFrame."""
    print(f"Fetching data for {device_name}...")
    json_string = generate(device_name)

    # Clean the string to remove markdown formatting
    if '```json' in json_string:
        json_string = json_string.split('```json')[1].split('```')[0]

    # Parse the JSON string into a Python dictionary
    data_dict = json.loads(json_string)

    # Separate and normalize the 'features' dictionary
    features_dict = data_dict.pop('features', {})

    # Create a single-row DataFrame for the new data by merging the dictionaries
    new_row_data = {**data_dict, **features_dict}
    new_row_df = pd.DataFrame([new_row_data])

    # Concatenate the new row to the existing DataFrame
    df = pd.concat([df, new_row_df], ignore_index=True)

    print(f"Successfully added and normalized {device_name} to the DataFrame.")
    return df

print("The 'get_and_append_phone_data' function has been updated to normalize the 'features' column.")

The 'get_and_append_phone_data' function has been updated to normalize the 'features' column.


## Get input data to the LLM from BigQuery


In [ ]:
from google.cloud import bigquery



# Construct a BigQuery client object.
client = bigquery.Client()

query = """
SELECT
  distinct upper(t1.manufacturer || ' ' || t1.marketing_name) AS user_input
FROM
  (
    SELECT
      device_producername as manufacturer,
      device_marketing_name as marketing_name,
      count(distinct customer_sk_user) AS max_footprint
    FROM
      `tnn-consumer-common-nx.ADM.subscription_hist`
    WHERE device_type IN ('PDA','CLAMSHELL','SMARTPHONE','BLOCK','SLIDER','RUGGED')
      and period_month_date is not null
    GROUP BY ALL
    -- Fjerner ORDER BY og LIMIT her, da de ikke er nødvendige for den ytre distinct SELECT
    -- og kan påvirke ytelsen for et stort datasett uten en LIMIT.
    -- Hvis du ønsker å beholde en form for sortering/limit for den *endelige* output,
    -- gjør det på det ytre SELECT-nivået.
  ) AS t1
WHERE
  upper(TRIM(t1.manufacturer||' '||t1.marketing_name)) NOT IN ( -- La inn TRIM() for sikkerhets skyld
    SELECT
      upper(TRIM(user_input)) -- La inn TRIM() for sikkerhets skyld
    FROM
      `tnn-consumer-common-nx.DataScience.device_features`
    WHERE
      user_input IS NOT NULL
  )
  and max_footprint >= 500;
"""

bq_df = client.query(query).to_dataframe()

print(f"DataFrame shape: {bq_df.shape}")
bq_df.head()

DataFrame shape: (4, 1)


,user_input
0,DORO LEVA L20 LEVA L21
1,DORO LEVA L10 LEVA L11
2,DORO LEVA L30 LEVA L31
3,DORO DORO 1355 DORO 1380 DORO 1378


## Save to BigQuery

### Subtask:
Functions that takes a DataFrame and writes it to the BigQuery table `tnn-consumer-common-nx.DataScience.device_features`. The function should automatically create the table if it does not exist and append new data if it already exists. This will be run in smaller baches.
- process_in_batches(df, batch_size, table_id)
- process_device(device_name)
- save_to_bigquery(df, table_id)


In [ ]:
from google.cloud import bigquery
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import json

# Updated save function to allow schema updates
def save_to_bigquery(df, table_id):
    """Saves a DataFrame to a BigQuery table, allowing schema updates."""
    client = bigquery.Client()
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND",
        create_disposition="CREATE_IF_NEEDED",
        schema_update_options=[
            bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION
        ]
    )
    job = client.load_table_from_dataframe(
        df, table_id, job_config=job_config
    )
    job.result()
    print(f"Successfully saved {len(df)} rows to {table_id}")

# 1. Define constants
BATCH_SIZE = 10
TABLE_ID = "tnn-consumer-common-nx.DataScience.device_features"

# 2. Worker function to process a single device
def process_device(device_name):
    """Fetches and processes data for a single device, with error handling."""
    try:
        print(f"Fetching data for {device_name}...")
        json_string = generate(device_name)

        if '```json' in json_string:
            json_string = json_string.split('```json')[1].split('```')[0]

        data_dict = json.loads(json_string)
        features_dict = data_dict.pop('features', {})

        # Combine dictionaries
        combined_dict = {**data_dict, **features_dict}

        # Standardize keys and values
        processed_dict = {}
        for k, v in combined_dict.items():
            key = k.lower().replace(' ', '_')
            # Convert device_name and manufacturer to uppercase
            value = str(v).upper() if k.lower() in ['device_name', 'manufacturer'] else str(v)
            processed_dict[key] = value

        # Manually add the original user_input to ensure it's never missing
        processed_dict['user_input'] = device_name

        return processed_dict

    except Exception as e:
        print(f"Error processing {device_name}: {e}")
        return None

# 3. Main processing loop
def process_in_batches(df, batch_size, table_id):
    all_results = []
    num_batches = (len(df) + batch_size - 1) // batch_size
    for i in range(0, len(df), batch_size):
        batch_df = df[i:i + batch_size]
        device_names = batch_df['user_input'].tolist()
        current_batch_num = i // batch_size + 1
        print(f"--- Processing batch {current_batch_num}/{num_batches} ---")

        batch_results = []
        with ThreadPoolExecutor(max_workers=batch_size) as executor:
            results = executor.map(process_device, device_names)
            # Filter out None results from failed API calls
            successful_results = [res for res in results if res is not None]
            batch_results.extend(successful_results)

        if batch_results:
            results_df = pd.DataFrame(batch_results)
            all_results.append(results_df)
            print(f"\n--- Saving batch {current_batch_num} to BigQuery ---")
            save_to_bigquery(results_df, table_id)
        else:
            print(f"--- Batch {current_batch_num} had no successful results to save. ---")

    print("\n--- Finished processing all batches. ---")
    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame() # Return empty df if no results

## Run the pipeline

In [ ]:
# Execute the pipeline
final_df = process_in_batches(bq_df, BATCH_SIZE, TABLE_ID)

print("\n--- Final Combined DataFrame ---")
final_df

--- Processing batch 1/1 ---
Fetching data for DORO LEVA L20  LEVA L21...
Fetching data for DORO LEVA L10  LEVA L11...
Fetching data for DORO LEVA L30  LEVA L31...
Fetching data for DORO DORO 1355 DORO 1380 DORO 1378...
Error processing DORO LEVA L10  LEVA L11: pop expected at most 1 argument, got 2
Error processing DORO LEVA L30  LEVA L31: pop expected at most 1 argument, got 2
Error processing DORO LEVA L20  LEVA L21: Unterminated string starting at: line 15 column 21 (char 362)
Error processing DORO DORO 1355 DORO 1380 DORO 1378: Unterminated string starting at: line 17 column 24 (char 412)
--- Batch 1 had no successful results to save. ---

--- Finished processing all batches. ---

--- Final Combined DataFrame ---


""
